In [1]:
import json
import duckdb
import spacy

In [2]:
FILE_PATH = "../local-data/comments-only.json"

In [3]:
with open(FILE_PATH) as f:
    data = json.load(f)

In [4]:
data[0]

{'comment_id': 'ED-2025-OPE-0944-17445',
 'docket_id': 'ED-2025-OPE-0944',
 'agency_code': 'ED',
 'title': 'Comment on FR Doc # 2026-01912',
 'comment': "To the Department of Education,<br/><br/>I write today out of deep concern about the ramifications of the proposed rulemaking under the Higher Education Act (HEA) relating to the designation of &ldquo;graduate&rdquo; versus &ldquo;professional&rdquo; programs for purposes of defining loan limits in the Federal Unsubsidized loan program. Specifically, the proposed rule will reduce the ability of institutions like Lewis &amp; Clark to train mental health professionals as well as teachers and other K-12 school staff. Reducing workforce training in these key areas will have ripple effects in rural, suburban, and urban communities that would otherwise be served by our graduates. I strongly urge the Department to define &ldquo;professional&rdquo; programs in ways that reflect today&rsquo;s economy and that build the workforce America needs 

In [5]:
import re
from html import unescape

def clean_html_entities_and_strip_tags(text: str) -> str:
    """
    - Replace &amp; with &
    - Replace &ldquo; and &rdquo; with straight quotes (")
    - Strip all HTML tags
    - Unescape other HTML entities safely
    """
    if text is None:
        return ""

    # 1) Replace specific named entities for curly quotes with straight quotes
    text = text.replace("&ldquo;", '"').replace("&rdquo;", '"')

    # 2) Replace ampersand entity (handle repeated/encoded forms)
    text = text.replace("&amp;", "&")

    # 3) Remove HTML tags (simple conservative regex)
    text = re.sub(r"<[^>]+>", " ", text)

    # 4) Unescape any remaining HTML entities (e.g., &nbsp;, &lt;, numeric entities)
    text = unescape(text)

    # 5) Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [6]:
import en_core_web_md
nlp = en_core_web_md.load()

In [7]:
comment = clean_html_entities_and_strip_tags(data[0]['comment'])
comment

'To the Department of Education, I write today out of deep concern about the ramifications of the proposed rulemaking under the Higher Education Act (HEA) relating to the designation of "graduate" versus "professional" programs for purposes of defining loan limits in the Federal Unsubsidized loan program. Specifically, the proposed rule will reduce the ability of institutions like Lewis & Clark to train mental health professionals as well as teachers and other K-12 school staff. Reducing workforce training in these key areas will have ripple effects in rural, suburban, and urban communities that would otherwise be served by our graduates. I strongly urge the Department to define "professional" programs in ways that reflect today’s economy and that build the workforce America needs for a stable and strong future. When Congress passed the One Big Beautiful Bill Act (OBBBA), it established higher annual and lifetime federal loan limits for students in professional graduate programs. Curre

In [8]:
doc = nlp(comment)

In [9]:
doc.ents

(the Department of Education,
 today,
 the Higher Education Act,
 Federal Unsubsidized,
 Lewis & Clark,
 K-12,
 Department,
 today,
 America,
 Congress,
 One,
 annual,
 Department,
 approximately 93%,
 K-12,
 Oregon,
 US,
 •,
 K-12,
 K-12,
 •,
 K-12,
 •,
 Federal Unsubsidized,
 annual,
 nearly four,
 Department,
 Robin Holmes-Sullivan,
 Lewis & Clark College)

In [11]:
def process(comment: str) -> str:
    text = clean_html_entities_and_strip_tags(comment)
    doc = nlp(text)
    return doc.ents

In [13]:
len(data)

17730

In [17]:
import random
random.seed(123)
sample_index = [random.randint(0, len(data)) for i in range(10)]

In [19]:
for i in sample_index:
    print(data[i]['comment'])
    print(process(data[i]['comment']))
    print("\n")


I appreciate the opportunity to comment on the U.S. Department of Education&rsquo;s Reimagining and Improving Student Education proposed rule. If finalized as proposed, the Doctor of Physical Therapy degree, or DPT, would be arbitrarily reclassified as a graduate degree with an annual federal loan borrowing cap of $20,500 and a lifetime borrowing limit of $100,000. Considering its longstanding recognition as a professional degree, this reclassification would significantly reduce the ability of future physical therapists to borrow federal student loans, threatening to limit the physical therapy workforce at a time of nationwide shortages and increasing demand for services. I strongly urge the department to support students pursuing careers as physical therapists by adopting a definition that reflects the profession&rsquo;s advanced level of education and status as a doctoring profession.<br/><br/>As for me, being raised in a low, poverty-stricken area where finances are a significant ma